# 白鷺祭 FLOW — MNIST学習
GPUランタイムを選択して実行。AIのコアは元ノートと同一。20epochは目安で品質保証ではありません。チェックポイントは各epochで更新されます。Colabの切断に備えてDriveに保存します。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!git clone https://github.com/ryotkkr-del/Flow-Matching.git /content/Flow-Matching
%cd /content/Flow-Matching
import torch
print(torch.__version__, "GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "ランタイム → ランタイムのタイプを変更 → GPU を選択してください"

## 1. 学習
元ノートのsample_path、TinyUNet、MSEを維持。Drive保存先が既にあれば続きから実行。学習後に出力画像を確認します。

In [ ]:
import os, subprocess, sys
checkpoint="/content/drive/MyDrive/FlowMatching/mnist.pt"
args=[sys.executable,"-m","ai.train","--epochs","20","--out",checkpoint]
if os.path.exists(checkpoint): args += ["--resume", checkpoint]
subprocess.run(args,check=True)

## 2. 実際の生成過程を保存
生成結果のPNGと、全時刻の画像・速度を出力。学習用checkpointはそのまま残します。

In [ ]:
import shutil
os.makedirs("checkpoints",exist_ok=True)
shutil.copyfile(checkpoint,"checkpoints/mnist.pt")
subprocess.run([sys.executable,"-m","ai.export","--steps","20","50","100","--seeds","42","43","44"],check=True)
from IPython.display import display, Image
for seed in [42,43,44]:
    display(Image(filename=f"demos/seed{seed}-steps50.png"))

## 3. 品質を確認してPCへ
12枚の数字を目視し、不鮮明なら上のepochsを30などにして再開。全時刻は実際のモデルの出力で、補間の見せかけではありません。ダウンロードしたzipの中身をPCのアプリ直下へ展開してください。

In [ ]:
subprocess.run([sys.executable,"-m","ai.verify","demos/seed42-steps50.json.gz"],check=True)
import zipfile
with zipfile.ZipFile("trained-exhibition.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write("checkpoints/mnist.pt")
    from pathlib import Path
    for p in Path("demos").glob("*"):
        if p.is_file(): z.write(p)
from google.colab import files
files.download("trained-exhibition.zip")